In [1]:
import numpy as np
import pandas as pd

from scformer.utils import *
from scformer.model import *
from warnings import filterwarnings
import random
import os
import torch
import torch.cuda as cuda
from scipy import sparse
import scanpy as sc
import anndata as ad

filterwarnings("ignore")
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

In [2]:
gene_cell = ad.read_mtx('data/mouse_retina/Gene_Cell.mtx')
gene_names = pd.read_csv('data/mouse_retina/Gene_names.tsv', sep='\t', header=None)
true_label = pd.read_csv('data/mouse_retina/Cell_type.tsv', sep='\t', header=None)

gene_cell.obs_names = gene_names[0]

RNA_matrix = gene_cell.X

cell_num = RNA_matrix.shape[1]
gene_num = RNA_matrix.shape[0]
cell_num

9383

In [3]:
# initial_pre = initial_clustering(RNA_matrix)
# cluster_ini_num = len(set(initial_pre))
# ini_p1 = [int(i) for i in initial_pre]

# partite the data into batches
indices, Node_Ids, dic = batch_select_whole(RNA_matrix)
n_batch = len(indices)
device = torch.device("cuda" if cuda.is_available() else "cpu")

Partitioning the data into batches. Please wait...


Processing Batches: 100%|██████████| 313/313 [00:39<00:00,  7.94it/s]


In [4]:
node_model = NDR_1(
    RNA_matrix=RNA_matrix,
    indices=indices,
    n_hid=104,
    n_heads=8,
    n_layers=3,
    labsm=0.1,
    lr=0.0005,
    wd=0.1,
    device=device,
    num_types=2,
    num_relations=2,
    epochs=100,
)
gnn, cell_emb, gene_emb, h = node_model.train_model(n_batch=n_batch)

The training process for the NodeDimensionReduction model has started. Please wait.
When the number of cells is less than or equal to 500, the resolution value should be set to 0.2.
When the number of cells is within the range of 500 to 5000, the resolution value should be set to 0.5.
When the number of cells is greater than 5000, the resolution value should be set to 0.8.
         Falling back to preprocessing with `sc.pp.pca` and default params.
Leiden clustering completed.
Starting GNN forward pass to compute embeddings...
GNN embedding computation completed.


100%|██████████| 100/100 [31:05<00:00, 18.65s/it]

The training for the NodeDimensionReduction model has been completed.


In [9]:
# 保存 NodeDimensionReduction 训练好的模型和聚类中心
node_model.save_model('node_dimension_reduction_model.pth')

In [10]:
# 加载 NodeDimensionReduction 模型和聚类中心
# 假设您知道 n_hid 和 num_clusters
loaded_node_model = NDR_1.load_model(
    file_path='node_dimension_reduction_model.pth',
    device=device,
    RNA_matrix=RNA_matrix,
    indices=indices
)

In [11]:
scformer_model = HGT_model(
    gnn=loaded_node_model.gnn,
    cluster_centers=loaded_node_model.cluster_centers,
    labsm=0.1,
    n_hid=10,
    device=device
)

# 预测并保存结果
ScFormer_result = scformer_model.predict(
    RNA_matrix=RNA_matrix,
    indices=indices,
    nodes_id=Node_Ids,  # 您的节点 ID 列表
    cell_size=30
)

Prediction Batches: 100%|██████████| 313/313 [00:10<00:00, 29.34it/s]


In [13]:
# Save numpy arrays to files
output_file = 'data/mouse_retina/output'
np.save(output_file + "/Node_Ids.npy", Node_Ids)
np.save(output_file + "/pred.npy", ScFormer_result['pred_label'])
np.save(output_file + "/cell_embedding.npy", ScFormer_result['cell_embedding'])